In [ ]:
!pip install transformers torch groq python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path
from datetime import datetime
from groq import Groq
from dotenv import load_dotenv
from transformers import pipeline
from google.colab import userdata

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

# GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GROQ_API_KEY = "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"
if (not GROQ_API_KEY):
  userdata.get('GROQ_API_KEY')

MODEL        = "llama-3.1-8b-instant"
SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────────────
# NLP LAYER — Zero Shot Cognitive Shift Tracker
# ─────────────────────────────────────────────

print("Loading NLP model... (first run downloads ~1.6gb)")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
print("NLP model ready.\n")

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive":  "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":     "taking action, making decisions, expressing control, planning, choosing"
}

def score_message(text):
    """NLP Layer — score a single message across affective, cognitive, agency."""
    candidate_labels = list(LABEL_DESCRIPTIONS.values())
    result = nlp_classifier(text, candidate_labels, multi_label=False)

    scores = {}
    for label, score in zip(result['labels'], result['scores']):
        for clean_name, description in LABEL_DESCRIPTIONS.items():
            if label == description:
                scores[clean_name] = round(score, 4)

    dominant = max(scores, key=scores.get)
    return scores, dominant


def compute_auc_summary(cognitive_shift):
    """
    Compute area under the curve across all scored messages.
    This is the unbiased whole-session picture —
    not influenced by what happened at the end.
    """
    if not cognitive_shift:
        return {}

    affective = [m['scores']['affective'] for m in cognitive_shift]
    cognitive = [m['scores']['cognitive'] for m in cognitive_shift]
    agency    = [m['scores']['agency']    for m in cognitive_shift]

    def trapz(values):
        if len(values) < 2:
            return round(values[0], 4)
        total = 0
        for i in range(1, len(values)):
            total += (values[i] + values[i-1]) / 2
        return round(total, 4)

    avg = lambda v: round(sum(v) / len(v), 4)

    avg_a = avg(affective)
    avg_c = avg(cognitive)
    avg_g = avg(agency)

    dominant      = max({'affective': avg_a, 'cognitive': avg_c, 'agency': avg_g}, key=lambda k: {'affective': avg_a, 'cognitive': avg_c, 'agency': avg_g}[k])
    last_dominant = cognitive_shift[-1]['dominant']
    recency_bias  = last_dominant != dominant

    return {
        "area_under_curve": {
            "affective": trapz(affective),
            "cognitive": trapz(cognitive),
            "agency":    trapz(agency)
        },
        "session_averages": {
            "affective": avg_a,
            "cognitive": avg_c,
            "agency":    avg_g
        },
        "dominant_mode":          dominant,
        "last_message_dominant":  last_dominant,
        "recency_bias_detected":  recency_bias,
        "recency_bias_warning": (
            f"Last message suggests '{last_dominant}' but overall session was '{dominant}'. "
            f"AI summary may be biased toward end of conversation."
        ) if recency_bias else "No recency bias detected."
    }


# ─────────────────────────────────────────────
# SESSION MANAGEMENT
# ─────────────────────────────────────────────

def get_session_id():
    """Generate a session ID based on existing session files."""
    existing = sorted(SESSIONS_DIR.glob("session_*.json"))
    return len(existing) + 1


def load_session(session_id):
    """Load an existing session JSON."""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def save_session(session_id, data):
    """Save session JSON to disk."""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def new_session(session_id):
    """Create a fresh session object."""
    return {
        "session_id":      session_id,
        "date":            datetime.now().strftime("%Y-%m-%d"),
        "session_type":    "brain_dump",
        "priorities":      [],
        "blockers":        [],
        "small_wins":      [],
        "insights":        [],
        "mood":            "",
        "cognitive_shift": []   # NLP layer data lives here
    }


# ─────────────────────────────────────────────
# RAG LAYER — Recency Based (last 3 sessions)
# ─────────────────────────────────────────────

def retrieve_past_context(current_session_id):
    """
    RAG Layer — pull the last 3 sessions and format
    them into a concise context block for the AI.
    """
    past_sessions = []

    for i in range(current_session_id - 1, max(0, current_session_id - 4), -1):
        data = load_session(i)
        if data:
            past_sessions.append(data)

    if not past_sessions:
        return "No previous sessions found."

    context_blocks = []

    for s in past_sessions:
        auc = s.get("cognitive_shift_summary", {})
        dominant = auc.get("dominant_mode", "unknown")
        recency_bias = auc.get("recency_bias_detected", False)

        block = f"""
--- Session {s['session_id']} ({s['date']}) ---
Mood: {s.get('mood', 'unknown')}
Priorities: {', '.join(s.get('priorities', [])) or 'none'}
Blockers: {', '.join(s.get('blockers', [])) or 'none'}
Small wins: {', '.join(s.get('small_wins', [])) or 'none'}
Insights: {', '.join(s.get('insights', [])) or 'none'}
Dominant emotional mode: {dominant}
Recency bias detected: {recency_bias}
""".strip()

        context_blocks.append(block)

    return "\n\n".join(context_blocks)


# ─────────────────────────────────────────────
# AI LAYER — Groq
# ─────────────────────────────────────────────

client = Groq(api_key=GROQ_API_KEY)

def get_ai_response(conversation, current_notes, past_context):
    """
    AI Layer — builds system prompt with RAG context
    and current notes, calls Groq, returns reply.
    """
    system_prompt = f"""You are Indy, an ADHD copilot. Be gentle, short, no guilt.
Respond in the same language as the user.

### PAST SESSIONS (for context)
{past_context}

### CURRENT SESSION NOTES
{json.dumps(current_notes, indent=2, ensure_ascii=False)}

After your reply, ALWAYS end with exactly:
### UPDATED_NOTES
{{full updated JSON object with English keys, do NOT include cognitive_shift key}}

Do not add extra text after the JSON.
"""

    messages = [{"role": "system", "content": system_prompt}] + conversation[-12:]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=600,
    )

    return response.choices[0].message.content.strip()


def extract_notes(reply, current_notes):
    """Parse the ### UPDATED_NOTES block from AI reply."""
    if "### UPDATED_NOTES" in reply:
        try:
            json_part = reply.split("### UPDATED_NOTES")[-1].strip()
            # Strip markdown code fences if present
            json_part = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part = re.sub(r"```$", "", json_part).strip()
            return json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"  → Warning: invalid JSON from model — keeping old notes. {e}")
            return current_notes
    else:
        print("  → Warning: model didn't include ### UPDATED_NOTES marker")
        return current_notes


def clean_reply(reply):
    """Strip the ### UPDATED_NOTES block from the reply shown to user."""
    if "### UPDATED_NOTES" in reply:
        return reply.split("### UPDATED_NOTES")[0].strip()
    return reply


# ─────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────

def main():
    session_id   = get_session_id()
    session_data = new_session(session_id)
    conversation = []
    turn_count   = 0

    print(f"\n{'='*55}")
    print(f"  Indy — ADHD Copilot")
    print(f"  Session #{session_id:03d} | {session_data['date']}")
    print(f"  NLP Cognitive Shift Tracker: Active")
    print(f"  RAG: pulling last 3 sessions")
    print(f"  Type 'quit' to end session")
    print(f"{'='*55}\n")

    print("Indy: Hey, I'm here. What's going on today?\n")

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ["quit", "exit", "bye"]:
            # ── Final cognitive shift summary ──
            if session_data["cognitive_shift"]:
                summary = compute_auc_summary(session_data["cognitive_shift"])
                session_data["cognitive_shift_summary"] = summary

                print("\n" + "="*55)
                print("SESSION SUMMARY — NLP Layer")
                print("="*55)
                print(f"Dominant mode:        {summary['dominant_mode']}")
                print(f"Recency bias:         {summary['recency_bias_detected']}")
                print(f"  → {summary['recency_bias_warning']}")
                print(f"Session averages:")
                print(f"  Affective:  {summary['session_averages']['affective']}")
                print(f"  Cognitive:  {summary['session_averages']['cognitive']}")
                print(f"  Agency:     {summary['session_averages']['agency']}")

            save_session(session_id, session_data)
            print(f"\nSession saved → sessions/session_{session_id:03d}.json")
            print("Indy: Take care of yourself. See you next time 💙\n")
            break

        # ── NLP LAYER — score this message ──
        scores, dominant = score_message(user_input)
        session_data["cognitive_shift"].append({
            "message_number": len(session_data["cognitive_shift"]) + 1,
            "text":           user_input,
            "scores":         scores,
            "dominant":       dominant
        })
        print(f"  [NLP] affective={scores['affective']} | cognitive={scores['cognitive']} | agency={scores['agency']} | dominant={dominant}")

        # ── RAG LAYER — retrieve past context ──
        past_context = retrieve_past_context(session_id)

        # ── AI LAYER — get response ──
        conversation.append({"role": "user", "content": user_input})

        try:
            reply = get_ai_response(conversation, session_data, past_context)
        except Exception as e:
            print(f"  API error: {e}")
            continue

        # ── Extract updated notes from reply ──
        updated_notes = extract_notes(reply, session_data)

        # Preserve cognitive_shift — AI should never overwrite it
        updated_notes["cognitive_shift"]    = session_data["cognitive_shift"]
        updated_notes["session_id"]         = session_id
        updated_notes["date"]               = session_data["date"]
        session_data                        = updated_notes

        clean = clean_reply(reply)
        print(f"\nIndy: {clean}\n")

        conversation.append({"role": "assistant", "content": clean})
        turn_count += 1

        # Save after every turn
        save_session(session_id, session_data)
        print(f"  ✓ Session saved (turn {turn_count})")
        print("-" * 55)


if __name__ == "__main__":
    main()

Loading NLP model... (first run downloads ~1.6gb)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
!python --version

In [ ]:
!pip install virtualenv
!virtualenv myenv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')